# BSPC-D-26-17774: revision re-run on the regenerated images

Main analysis uses the **regenerated** images: every ECG recording rendered twice (pseudo-coloured and colourless) at a fixed image size on a **fixed time axis**, so absolute QT and RR are preserved as distances. Labels, participants, drugs and timepoints come from `mapping_table_with_outcome_and_distance.csv`.

| Reviewer point | What this notebook does |
|---|---|
| R1.2 participant-grouped evaluation | 5-fold CV with all of a participant's images in one fold (primary). Also the original image-level split, and a split keeping each participant-period-timepoint triplicate together. |
| R1.3 matched renderings | Same recordings in both renderings, one shared fold assignment; exclusions reported by rendering and class. |
| R1.4 information in the overlay | QT/heart-rate logistic baseline; greyscale version of the coloured images; **old R-to-R-stretched images as a third condition** (no absolute time) vs fixed-axis colourless (time, no threshold) vs fixed-axis coloured (time and threshold). |
| R1.5 what the probe shows | Image-only, image + prompt, prompt-only probes; standardised features; no constant "no-context" block. |
| R1.6 statistics | Participant-cluster bootstrap CIs for paired differences, Holm adjustment, nested (inner-CV) prompt selection. |
| R1.7 operating points | Pooled out-of-fold confusion matrices, precision-recall, thresholds chosen on training data only, specificity at matched sensitivity. |
| R1.8 reproducibility | Seeds, split manifest, model revision, package versions; the original protocol re-run on the **old** images to reproduce the published numbers. |
| R2.5, R2.6, R2.13 | Cohort table, hardware and run time, exclusion report. |

**How to run**: *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all* and allow Drive access. Results go to `My Drive/BSPC_revision_rerun` (roughly 30-50 minutes).

**Check first when it finishes**
- *Section 1*: image counts per rendering, how many recordings matched, and whether the two label sources (`QT_nomogram_distance_ms` and `outcome`) agree.
- *Section 10*: whether the old images link to the same recordings (third condition runs only if they do).
- *Section 11*: the original protocol on the old images should reproduce the published counts and MCCs.

**Important:** the colour prompts are unchanged ("a T wave with a high amount of blue/green" vs "...orange/red"). If the new colour scale uses different colours for beats beyond the threshold, edit `COLOUR_PROMPTS` in Section 3 before running.

In [ ]:
# Install (Colab already has torch, scikit-learn, pandas)
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'open_clip_torch'], check=True)

## Config

In [ ]:
SMOKE_TEST = False   # True = synthetic data and a stand-in model, only for testing the code

DRIVE_ROOT = '/content/drive/MyDrive'
ECG2026 = f'{DRIVE_ROOT}/Colab Notebooks/2026/ECGRDVQ'

# Regenerated images (fixed time axis): a .zip or folder per rendering; file names start with the ECG recording number
IMAGE_SOURCES = {
    'colour':     f'{ECG2026}/images/study/Colour_Heartbeat.zip',
    'colourless': f'{ECG2026}/images/study/Colourless_Heartbeat.zip',
}
METADATA_CSV = f'{ECG2026}/metadata/mapping_table_with_outcome_and_distance.csv'
META_COLS = {'id': 'ECG_recording_number', 'participant': 'patient_id', 'drug': 'EXTRT', 'period': 'VISIT',
             'timepoint': 'TPT', 'qt': 'QT', 'rr': 'RR', 'hr': 'HR',
             'distance': 'QT_nomogram_distance_ms', 'outcome': 'outcome'}
LABEL_FROM = 'distance'   # 'distance': high risk if QT_nomogram_distance_ms >= 0 (on/above the line); 'outcome': use the outcome column

# Images of the submitted paper (stretched R to R, labels in 'At risk'/'No risk' folders)
RUN_OLD = True
OLD_SB = f'{DRIVE_ROOT}/Colab Notebooks/Images/single heartbeat/Benchmark'
OLD_SOURCES = {
    'colour':     f'{OLD_SB}/colour/All_Data/All.zip',
    'colourless': f'{OLD_SB}/Nocolour/All_Data/All.zip',
}

LOCAL_DATA = '/content/data'     # zips are copied and unpacked here
OUT_DIR = f'{DRIVE_ROOT}/BSPC_revision_rerun'

MODEL_ID = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
SEED = 42
N_SPLITS = 5          # outer CV folds
N_INNER = 4           # inner folds (threshold choice and prompt selection)
N_BOOT = 2000         # participant-cluster bootstrap replicates
ALPHA = 0.5           # colour/morphology fusion weight (as in the paper)
STANDARDISE = True    # z-score features before the probe
RUN_GREYSCALE = True  # extra rendering: coloured images converted to greyscale
RUN_INNER = True      # inner CV for thresholds and nested prompt selection
BATCH_SIZE = 64

In [ ]:
# Synthetic data for SMOKE_TEST only (skipped on a real run)
if SMOKE_TEST:
    import os, shutil, numpy as np, pandas as pd
    from PIL import Image
    rng = np.random.default_rng(0)
    root = '/tmp/smoke_data'
    shutil.rmtree(root, ignore_errors=True)
    rows = []
    drugs = ['Placebo', 'Dofetilide', 'Quinidine', 'Ranolazine', 'Verapamil HCL']
    tpts = [-0.5, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 5, 6, 7, 8, 12, 14, 24]
    for s in range(22):
        base_qt, base_rr = rng.normal(400, 15), rng.normal(1000, 80)
        sens = rng.uniform(0.3, 1.6)
        order = rng.permutation(5)
        for pi, d in enumerate(drugs):
            eff = {'Placebo': 0, 'Dofetilide': 70, 'Quinidine': 55, 'Ranolazine': 10, 'Verapamil HCL': 3}[d]
            for t in tpts:
                conc = 0 if t < 0 else np.exp(-((t - 2.5) ** 2) / 10)
                qt_t = base_qt + sens * eff * conc + rng.normal(0, 8)
                rr_t = base_rr + rng.normal(0, 40)
                for rep in range(3):
                    qt, rr = qt_t + rng.normal(0, 3), rr_t + rng.normal(0, 10)
                    hr = 60000 / rr
                    rows.append({'EGREFID': f'x{len(rows)}', 'patient_id': 1001 + s, 'SEX': 'F' if s % 2 else 'M',
                                 'AGE': int(rng.integers(20, 40)), 'HGHT': 170.0, 'WGHT': 70.0, 'RACE': 'WHITE',
                                 'EXTRT': d, 'VISIT': f'PERIOD-{order[pi] + 1}-DOSING', 'TPT': t,
                                 'HR': hr, 'RR': rr, 'QT': qt,
                                 'QT_nomogram_distance_ms': qt - (480 - 1.2 * (hr - 60))})
    meta = pd.DataFrame(rows)
    meta.insert(0, 'ECG_recording_number', rng.permutation(len(meta)) + 1)
    meta['outcome'] = np.where(meta.QT_nomogram_distance_ms >= 0, 'risk', 'no risk')
    meta.loc[meta.index[:13], ['QT', 'QT_nomogram_distance_ms', 'outcome']] = np.nan   # like the real table
    os.makedirs(root, exist_ok=True)
    meta.sample(frac=1, random_state=1).to_csv(f'{root}/metadata.csv', index=False)
    pos = np.where(meta.QT_nomogram_distance_ms >= 0)[0]
    keep = np.sort(np.concatenate([pos, np.arange(13), rng.choice(np.setdiff1d(np.arange(13, len(meta)), pos), 900, replace=False)]))
    subj_pattern = {s: rng.uniform(0, 1, 3) for s in meta.patient_id.unique()}
    def draw(r, coloured, stretched):
        img = np.zeros((16, 16, 3)); img[...] = subj_pattern[r.patient_id] * 0.5
        excess = 0 if pd.isna(r.QT_nomogram_distance_ms) else r.QT_nomogram_distance_ms / 60
        qt_ = 400 if pd.isna(r.QT) else r.QT
        width = 16 if stretched else int(np.clip(r.RR / 1600 * 16, 6, 16))
        qt_px = int(np.clip(qt_ / (r.RR if stretched else 1600) * 16, 1, 15))
        img[:, :qt_px, :] += 0.3
        if coloured:
            img[:, 8:width, 0] += np.clip(0.5 + excess, 0, 1)
        img[:, width:, :] = 0.9
        return Image.fromarray((np.clip(img + rng.normal(0, 0.08, img.shape), 0, 1) * 255).astype('uint8'))
    for rend, sub, coloured in [('colour', 'beat_colour', True), ('colourless', 'beat_line', False)]:
        d = f'{root}/new_{rend}/{sub}'; os.makedirs(d, exist_ok=True)
        for i in keep:
            r = meta.iloc[i]
            if rend == 'colourless' and rng.random() < 0.01:
                continue
            draw(r, coloured, False).save(f'{d}/{r.ECG_recording_number}.png')
        shutil.make_archive(f'{root}/{rend}_new', 'zip', f'{root}/new_{rend}')
    for rend, coloured in [('colour', True), ('colourless', False)]:
        for lab in ['At risk', 'No risk']:
            os.makedirs(f'{root}/old_{rend}/All/{lab}', exist_ok=True)
        for i in keep:
            r = meta.iloc[i]
            if rend == 'colourless' and r.QT_nomogram_distance_ms < 0 and rng.random() < 0.06:
                continue
            if pd.isna(r.QT_nomogram_distance_ms):
                continue
            lab = 'At risk' if r.QT_nomogram_distance_ms >= 0 else 'No risk'
            draw(r, coloured, True).save(f'{root}/old_{rend}/All/{lab}/{r.ECG_recording_number}_Heartbeat_1.png')
        shutil.make_archive(f'{root}/{rend}_old', 'zip', f'{root}/old_{rend}')
    IMAGE_SOURCES = {'colour': f'{root}/colour_new.zip', 'colourless': f'{root}/colourless_new.zip'}
    OLD_SOURCES = {'colour': f'{root}/colour_old.zip', 'colourless': f'{root}/colourless_old.zip'}
    METADATA_CSV = f'{root}/metadata.csv'
    LOCAL_DATA = '/tmp/smoke_local'
    OUT_DIR = '/tmp/smoke_out'
    N_BOOT = 200
    print('Synthetic data written:', len(keep), 'recordings')

## Setup

In [ ]:
import os, re, json, time, glob, random, platform, datetime, zlib, warnings
import numpy as np
import pandas as pd
from PIL import Image
import sklearn
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from scipy.stats import rankdata

if IN_COLAB and not SMOKE_TEST:
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

try:
    display
except NameError:
    def display(x):
        print(x.to_string() if hasattr(x, 'to_string') else x)

TIMINGS = []
class timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a):
        dt = time.time() - self.t
        TIMINGS.append({'step': self.name, 'seconds': round(dt, 1)})
        print(f'[{self.name}] {dt:.1f} s')

def save(df, name):
    df.to_csv(os.path.join(OUT_DIR, name), index=False)
    print('saved', name)

## 1. Recordings, labels and images (R1.1, R1.3, R2.13)

In [ ]:
import zipfile, shutil
AT_RISK = re.compile(r'^at[ _-]?risk$', re.I)
NO_RISK = re.compile(r'^no[ _-]?risk$', re.I)

def prepare_source(name, src):
    if not os.path.exists(src):
        raise FileNotFoundError(f'{name}: {src} not found - check the paths in Config')
    if src.lower().endswith('.zip'):
        dest = os.path.join(LOCAL_DATA, name)
        if not (os.path.isdir(dest) and any(os.scandir(dest))):
            os.makedirs(dest, exist_ok=True)
            local_zip = os.path.join(LOCAL_DATA, f'{name}.zip')
            shutil.copy(src, local_zip)
            with zipfile.ZipFile(local_zip) as z:
                z.extractall(dest)
            print(f'{name}: unpacked {os.path.basename(src)}')
        return dest
    return src

def list_pngs(root):
    rows = []
    for dirpath, _, files in os.walk(root):
        if '__MACOSX' in dirpath:
            continue
        base = os.path.basename(dirpath)
        fy = 1 if AT_RISK.match(base) else 0 if NO_RISK.match(base) else np.nan
        for f in files:
            if f.lower().endswith('.png') and not f.startswith('._'):
                m_ = re.match(r'^(\d+)', f)
                rows.append({'path': os.path.join(dirpath, f), 'file_name': f,
                             'rec': int(m_.group(1)) if m_ else np.nan, 'y_folder': fy})
    if not rows:
        raise FileNotFoundError(f'No PNG files under {root}')
    return pd.DataFrame(rows)

def clean(df, name):
    no_num = df.rec.isna().sum()
    if no_num:
        print(f'  {name}: {no_num} files without a leading recording number are ignored')
    df = df.dropna(subset=['rec']).copy()
    df['rec'] = df.rec.astype(int)
    dup = df.rec.duplicated().sum()
    if dup:
        print(f'  {name}: {dup} duplicate recording numbers; keeping the first file')
    return df.drop_duplicates('rec').reset_index(drop=True)

with timer('inventory (new images)'):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    NEW = {r: clean(list_pngs(prepare_source(f'new_{r}', s)), r) for r, s in IMAGE_SOURCES.items()}
for r, df in NEW.items():
    print(f'{r}: {len(df)} images')

# Metadata and labels
C = META_COLS
META = pd.read_csv(METADATA_CSV)
print('Metadata:', META.shape)
dup = META[C['id']].duplicated().sum()
if dup:
    print(f'{dup} duplicate recording numbers in the metadata; keeping the first row')
META = META.drop_duplicates(C['id']).copy()
META['rec'] = META[C['id']].astype(int)
labels = {}
if C['distance'] in META:
    dist = pd.to_numeric(META[C['distance']], errors='coerce')
    labels['distance'] = (dist >= 0).astype(float).where(dist.notna())      # missing QT -> no label
if C['outcome'] in META:
    print('outcome values:', META[C['outcome']].value_counts(dropna=False).to_dict())
    labels['outcome'] = META[C['outcome']].map(
        lambda s: np.nan if pd.isna(s) else (0.0 if str(s).strip().lower().startswith('no') else 1.0))
if len(labels) == 2:
    AGREE = pd.crosstab(labels['distance'].rename('distance >= 0'), labels['outcome'].rename('outcome = risk'),
                        dropna=False)
    display(AGREE)
    AGREE.to_csv(os.path.join(OUT_DIR, 'label_agreement.csv'))
    both = labels['distance'].notna() & labels['outcome'].notna()
    n_diff = int((labels['distance'][both] != labels['outcome'][both]).sum())
    if n_diff:
        print(f'NOTE: {n_diff} recordings differ between the two label sources')
META['y'] = labels[LABEL_FROM] if LABEL_FROM in labels else list(labels.values())[0]
print('Label source:', LABEL_FROM, '| high-risk:', int((META.y == 1).sum()), '| low-risk:', int((META.y == 0).sum()),
      '| no label (e.g. missing QT):', int(META.y.isna().sum()), '| total:', len(META))

# Match renderings and metadata on the recording number
m = NEW['colour'][['rec', 'path', 'file_name']].merge(
    NEW['colourless'][['rec', 'path']], on='rec', how='outer', suffixes=('_colour', '_colourless'), indicator=True)
m = m.merge(META[['rec', 'y']], on='rec', how='left')
yk = lambda s: (int((s.y == 1).sum()), int((s.y == 0).sum()), int(s.y.isna().sum()))
excl = []
for tag, sub in [('colour image only (no colourless)', m[m._merge == 'left_only']),
                 ('colourless image only (no colour)', m[m._merge == 'right_only']),
                 ('both images but no label (missing QT or no metadata row)', m[(m._merge == 'both') & m.y.isna()])]:
    h, l, u = yk(sub)
    excl.append({'reason': tag, 'high_risk': h, 'low_risk': l, 'unknown_label': u})
no_img = META[~META.rec.isin(m.rec)]
excl.append({'reason': 'metadata row with no image', 'high_risk': int((no_img.y == 1).sum()),
             'low_risk': int((no_img.y == 0).sum()), 'unknown_label': 0})
MATCHED = m[(m._merge == 'both') & m.y.notna()].drop(columns=['_merge', 'y'])
MATCHED = MATCHED.merge(META, on='rec', how='left')
MATCHED['y'] = MATCHED.y.astype(int)
excl.append({'reason': 'ANALYSED (both renderings + metadata)', 'high_risk': int(MATCHED.y.sum()),
             'low_risk': int((MATCHED.y == 0).sum()), 'unknown_label': 0})
EXCL = pd.DataFrame(excl)
display(EXCL)
save(EXCL, 'data_exclusions_by_rendering.csv')

MATCHED['participant'] = MATCHED[C['participant']].astype(str)
MATCHED['drug'] = MATCHED[C['drug']].astype(str) if C['drug'] in MATCHED else 'NA'
MATCHED['period'] = MATCHED[C['period']].astype(str) if C['period'] in MATCHED else MATCHED['drug']
MATCHED['timepoint'] = MATCHED[C['timepoint']].astype(str) if C['timepoint'] in MATCHED else 'NA'
MATCHED['QT'] = pd.to_numeric(MATCHED[C['qt']], errors='coerce') if C['qt'] in MATCHED else np.nan
MATCHED['RR'] = pd.to_numeric(MATCHED[C['rr']], errors='coerce') if C['rr'] in MATCHED else np.nan
if C['hr'] in MATCHED:
    MATCHED['HR'] = pd.to_numeric(MATCHED[C['hr']], errors='coerce')
else:
    MATCHED['HR'] = np.where(MATCHED.RR.median() < 10, 60 / MATCHED.RR, 60000 / MATCHED.RR)
MATCHED['cluster'] = MATCHED.participant + '|' + MATCHED.period + '|' + MATCHED.timepoint
MATCHED = MATCHED.sort_values(['y', 'rec'], ascending=[False, True]).reset_index(drop=True)
Y = MATCHED.y.values
N = len(Y)
GROUPS_AVAILABLE = True
CLIN, COLS = META, {'participant': C['participant']}   # used for the cohort table
save(MATCHED[['rec', 'file_name', 'y', 'participant', 'drug', 'period', 'timepoint', 'QT', 'RR', 'HR']],
     'beat_metadata.csv')
print(f'Analysed: {N} recordings, {int(Y.sum())} high-risk ({Y.mean():.1%}), {MATCHED.participant.nunique()} participants')

## 2. Data hierarchy and cohort (R1.1, R1.2, R2.5)

In [ ]:
# Data hierarchy (R1.1, R1.2): how images and positives are spread over participants, drugs, timepoints
if GROUPS_AVAILABLE:
    per_part = (MATCHED.groupby('participant')
                .agg(images=('y', 'size'), high_risk=('y', 'sum'), drugs=('drug', 'nunique'),
                     timepoint_clusters=('cluster', 'nunique'))
                .reset_index().sort_values('high_risk', ascending=False))
    display(per_part)
    save(per_part, 'hierarchy_per_participant.csv')
    by_drug = MATCHED.groupby('drug').agg(images=('y', 'size'), high_risk=('y', 'sum')).reset_index()
    display(by_drug)
    save(by_drug, 'hierarchy_by_drug.csv')
    print(f"{MATCHED.participant.nunique()} participants; {(per_part.high_risk > 0).sum()} have at least one high-risk beat; "
          f"{MATCHED.cluster.nunique()} participant-drug-timepoint clusters "
          f"(median {MATCHED.groupby('cluster').size().median():.0f} images each)")

    # How alike are images from the same timepoint cluster (the triplicate ECGs)?
    g = MATCHED.groupby('cluster')
    within = g.QT.agg(lambda s: s.max() - s.min())[g.size() > 1]
    mixed = (g.y.nunique() > 1).sum()
    rng_ = np.random.default_rng(SEED)
    a, b = rng_.integers(0, N, 5000), rng_.integers(0, N, 5000)
    diff_cluster = MATCHED.cluster.values[a] != MATCHED.cluster.values[b]
    random_pairs = np.abs(MATCHED.QT.values[a] - MATCHED.QT.values[b])[diff_cluster]
    TRIP = pd.DataFrame([{
        'clusters_with_2plus_images': int(len(within)),
        'median_QT_range_within_cluster_ms': float(np.nanmedian(within)),
        'median_abs_QT_difference_random_pairs_ms': float(np.nanmedian(random_pairs)),
        'clusters_with_mixed_labels': int(mixed),
    }])
    display(TRIP)
    save(TRIP, 'triplicate_similarity.csv')

    # Cohort characteristics (R2.5), one row per participant
    if CLIN is not None:
        demo_cols = [c for c in CLIN.columns if c.strip().upper() in ('SEX', 'AGE', 'HGHT', 'WGHT', 'RACE', 'ETHNIC', 'BMI')]
        if demo_cols:
            per = CLIN.groupby(COLS['participant'])[demo_cols].first()
            rows = []
            for c in demo_cols:
                v = pd.to_numeric(per[c], errors='coerce')
                if v.notna().mean() > 0.9:
                    rows.append({'variable': c, 'summary': f'mean {v.mean():.1f} (SD {v.std():.1f}), range {v.min():.0f}-{v.max():.0f}'})
                else:
                    rows.append({'variable': c, 'summary': ', '.join(f'{k}: {n}' for k, n in per[c].value_counts().items())})
            COHORT = pd.DataFrame(rows)
            display(COHORT)
            save(COHORT, 'cohort_characteristics.csv')

## 3. Model, prompts and embeddings

In [ ]:
COLOUR_PROMPTS = ["a T wave with a high amount of blue/green",
                  "a T wave with a high amount of orange/red"]
LABEL_SETS = {   # exactly as in the original notebooks, including the line break in one L3 prompt
    "No Label": ["Normal", "Abnormal"],
    "L1": ["Normal T wave", "Abnormal T wave"],
    "L2": ["Normal", "Flat", "Large", "Inverted", "Biphasic", "Notched"],
    "L3": ["Normal T wave",
           "Abnormal T wave with flat amplitude",
           "Abnormal large T wave, taller than initial signal",
           "Abnormal large T wave, broader and larger than\n         initial signal",
           "Abnormal T wave with a minor inversion",
           "Abnormal T wave with a deep inversion",
           "Abnormal T wave with a giant inversion",
           "Abnormal Biphasic T wave(Type A)",
           "Abnormal Biphasic T wave(Type B)",
           "Abnormal Bifid or Notched T wave"],
}

ENV = {'date': datetime.datetime.now().isoformat(timespec='seconds'), 'python': platform.python_version(),
       'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
       'model_id': MODEL_ID, 'seed': SEED, 'standardise': STANDARDISE, 'n_splits': N_SPLITS,
       'n_inner': N_INNER, 'n_boot': N_BOOT, 'alpha': ALPHA, 'smoke_test': SMOKE_TEST}

if SMOKE_TEST:
    PROJ = np.random.default_rng(1).normal(size=(16 * 16 * 3, 512))
    LOGIT_SCALE = 100.0
    def encode_texts(texts):
        out = np.stack([np.random.default_rng(zlib.crc32(t.encode())).normal(size=512) for t in texts])
        return out / np.linalg.norm(out, axis=1, keepdims=True)
    def embed_images(paths, greyscale=False):
        out, failed = [], []
        for p in paths:
            im = Image.open(p)
            im = im.convert('L').convert('RGB') if greyscale else im.convert('RGB')
            v = (np.asarray(im.resize((16, 16)), float) / 255).ravel() @ PROJ
            out.append(v / np.linalg.norm(v))
        return np.array(out), failed
    ENV['device'] = 'cpu (smoke test stand-in model)'
else:
    import torch, open_clip
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, _, preprocess_val = open_clip.create_model_and_transforms(MODEL_ID)
    tokenizer = open_clip.get_tokenizer(MODEL_ID)
    model.eval().to(DEVICE)
    LOGIT_SCALE = float(model.logit_scale.exp().item())

    def encode_texts(texts):
        with torch.no_grad():
            t = model.encode_text(tokenizer(texts).to(DEVICE))
            t = t / t.norm(dim=-1, keepdim=True)
        return t.float().cpu().numpy()

    def embed_images(paths, greyscale=False):
        out, failed = [], []
        dim = model.visual.output_dim if hasattr(model.visual, 'output_dim') else 512
        for i in range(0, len(paths), BATCH_SIZE):
            batch, ok = [], []
            for j, p in enumerate(paths[i:i + BATCH_SIZE]):
                try:
                    im = Image.open(p)
                    im = im.convert('L').convert('RGB') if greyscale else im.convert('RGB')
                    batch.append(preprocess_val(im)); ok.append(j)
                except Exception as e:
                    failed.append({'path': p, 'error': str(e)})
            feats = np.full((len(paths[i:i + BATCH_SIZE]), dim), np.nan, dtype=np.float32)
            if batch:
                with torch.no_grad():
                    f = model.encode_image(torch.stack(batch).to(DEVICE))
                    f = f / f.norm(dim=-1, keepdim=True)
                feats[ok] = f.float().cpu().numpy()
            out.append(feats)
            if (i // BATCH_SIZE) % 20 == 0:
                print(f'  {i + len(batch)}/{len(paths)}')
        return np.concatenate(out), failed

    ENV.update({'torch': torch.__version__, 'open_clip': open_clip.__version__, 'device': DEVICE,
                'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none',
                'logit_scale': LOGIT_SCALE})
    try:
        from huggingface_hub import HfApi
        ENV['model_revision_sha'] = HfApi().model_info(MODEL_ID.replace('hf-hub:', '')).sha
    except Exception as e:
        ENV['model_revision_sha'] = f'unavailable ({e})'
print(ENV)

T_C = encode_texts(COLOUR_PROMPTS)
T_M = {L: encode_texts(p) for L, p in LABEL_SETS.items()}

In [ ]:
# Embeddings (cached in OUT_DIR, so a restart does not redo them)
def cached_embeddings(name, paths, greyscale=False):
    f = os.path.join(OUT_DIR, f'embeddings_{name}.npz')
    names = np.array([os.path.basename(p) for p in paths])
    if os.path.exists(f):
        z = np.load(f, allow_pickle=True)
        if len(z['names']) == len(names) and (z['names'] == names).all():
            print(f'{name}: loaded cached embeddings')
            return z['emb'], list(z['failed'])
    with timer(f'embed {name} ({len(paths)} images)'):
        emb, failed = embed_images(list(paths), greyscale=greyscale)
    np.savez(f, emb=emb, names=names, failed=np.array(failed, dtype=object))
    return emb, failed

E, FAILED = {}, []
for r in ['colour', 'colourless']:
    E[r], fl = cached_embeddings(r, MATCHED[f'path_{r}'].values)
    FAILED += [dict(x, rendering=r) for x in fl]
if RUN_GREYSCALE:
    E['colour_greyscale'], fl = cached_embeddings('colour_greyscale', MATCHED.path_colour.values, greyscale=True)
    FAILED += [dict(x, rendering='colour_greyscale') for x in fl]

OLD_INV, E_OLD = {}, {}
if RUN_OLD:
    try:
        for r, s in OLD_SOURCES.items():
            df = list_pngs(prepare_source(f'old_{r}', s))
            df = df[df.y_folder.notna()].drop_duplicates('file_name')
            df['y'] = df.y_folder.astype(int)
            OLD_INV[r] = df.sort_values(['y', 'file_name'], ascending=[False, True]).reset_index(drop=True)
            print(f'old {r}: {len(df)} images ({df.y.sum()} high-risk)')
            E_OLD[r], fl = cached_embeddings(f'old_{r}', OLD_INV[r].path.values)
            FAILED += [dict(x, rendering=f'old_{r}') for x in fl]
    except FileNotFoundError as e:
        print('Old images not found, old-image sections will be skipped:', e)
        OLD_INV, E_OLD = {}, {}

FAILED = pd.DataFrame(FAILED)
print('Images that failed to load:', len(FAILED))
save(FAILED if len(FAILED) else pd.DataFrame(columns=['path', 'error', 'rendering']), 'failed_images.csv')
bad = np.zeros(N, bool)
for r in E:
    bad |= np.isnan(E[r]).any(1)
if bad.any():
    print(f'Dropping {bad.sum()} recordings with a failed image')
    MATCHED = MATCHED[~bad].reset_index(drop=True)
    E = {r: e[~bad] for r, e in E.items()}
    Y = MATCHED.y.values
    N = len(Y)

## 4. Prompt probabilities and zero-shot results

In [ ]:
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def is_normal_prompt(s):   # same mapping to the binary label as the original map_to_binary()
    l = s.lower()
    return ('normal' in l and 'abnormal' not in l) or l.startswith('normal')

def prompt_blocks(Emb):
    Pc = softmax(LOGIT_SCALE * Emb @ T_C.T)
    out = {'colour': Pc}
    for L, labels in LABEL_SETS.items():
        Pm = softmax(LOGIT_SCALE * Emb @ T_M[L].T)
        w_bg = Pc[:, 0] * ALPHA + (1 - ALPHA)
        w_or = Pc[:, 1] * ALPHA + (1 - ALPHA)
        W = np.stack([w_bg if 'Normal' in lab else w_or for lab in labels], axis=1)  # original fusion rule
        Mx = Pm * W
        out[f'morph[{L}]'] = Pm
        out[f'both[{L}]'] = Mx / Mx.sum(1, keepdims=True)
    return out

def fast_auc(y, s):
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def metrics(y, pred, score=None):
    y = np.asarray(y).astype(int); pred = np.asarray(pred).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    tn = int(((y == 0) & (pred == 0)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * ppv * sens / (ppv + sens) if (ppv + sens) > 0 else 0.0
    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / den if den > 0 else 0.0
    out = dict(TP=tp, FN=fn, TN=tn, FP=fp, Sens=sens, Spec=spec, BA=(sens + spec) / 2, PPV=ppv, F1=f1, MCC=mcc)
    if score is not None:
        score = np.asarray(score, float)
        out['ROC_AUC'] = fast_auc(y, score)
        out['AUPRC'] = average_precision_score(y, score) if 0 < y.sum() < len(y) else np.nan
    return out

STRAT_NAMES = {'morph': 'morphology only (C̄+M)', 'both': 'colour + morphology (C+M)',
               'colour': 'colour only (C+M̄)', 'none': 'no prompt (C̄+M̄): constant "low risk"'}

def zero_shot_table(Emb, y):
    B = prompt_blocks(Emb)
    rows = []
    for L, labels in LABEL_SETS.items():
        nm = np.array([is_normal_prompt(s) for s in labels])
        for key in ['morph', 'both']:
            P = B[f'{key}[{L}]']
            pred = (~nm[P.argmax(1)]).astype(int)
            score = P[:, ~nm].sum(1)          # probability of the high-risk class (all non-Normal prompts)
            rows.append({'label_set': L, 'strategy': STRAT_NAMES[key], **metrics(y, pred, score)})
        rows.append({'label_set': L, 'strategy': STRAT_NAMES['colour'],
                     **metrics(y, B['colour'].argmax(1), B['colour'][:, 1])})
        rows.append({'label_set': L, 'strategy': STRAT_NAMES['none'],
                     **metrics(y, np.zeros_like(y), np.full(len(y), 0.5))})
    return pd.DataFrame(rows)

ZS = []
for r in ['colour', 'colourless']:
    t = zero_shot_table(E[r], Y); t.insert(0, 'rendering', r); ZS.append(t)
ZS = pd.concat(ZS, ignore_index=True)
display(ZS.round(3))
save(ZS, 'zero_shot_matched.csv')

## 5. Shared folds (R1.2, R1.3)

In [ ]:
SCHEMES, GROUPS = {}, {}
SCHEMES['image_stratified'] = np.empty(N, int)
for k, (_, te) in enumerate(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(N), Y)):
    SCHEMES['image_stratified'][te] = k
GROUPS['image_stratified'] = np.arange(N)
if GROUPS_AVAILABLE:
    for name, col in [('participant_grouped', 'participant'), ('timepoint_grouped', 'cluster')]:
        g = MATCHED[col].values
        SCHEMES[name] = np.empty(N, int)
        for k, (_, te) in enumerate(StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(N), Y, g)):
            SCHEMES[name][te] = k
        GROUPS[name] = g
PRIMARY = 'participant_grouped' if GROUPS_AVAILABLE else 'image_stratified'
print('Primary scheme:', PRIMARY)

fold_summary = []
for s, f in SCHEMES.items():
    for k in range(N_SPLITS):
        te = f == k
        row = {'scheme': s, 'fold': k + 1, 'images': int(te.sum()), 'high_risk': int(Y[te].sum())}
        if GROUPS_AVAILABLE:
            row['participants'] = MATCHED.participant[te].nunique()
        fold_summary.append(row)
FOLDS = pd.DataFrame(fold_summary)
display(FOLDS)
save(FOLDS, 'fold_summary.csv')
manifest = MATCHED[['file_name', 'y'] + (['participant', 'drug', 'timepoint'] if GROUPS_AVAILABLE else [])].copy()
for s, f in SCHEMES.items():
    manifest[f'fold_{s}'] = f + 1
save(manifest, 'split_manifest.csv')

## 6. Linear probes (R1.5, R1.7)

In [ ]:
CONTEXT = ['image_only', 'img+colour'] + [f'img+morph[{L}]' for L in LABEL_SETS] + [f'img+both[{L}]' for L in LABEL_SETS]
PROMPT_ONLY = ['prompt:colour'] + [f'prompt:morph[{L}]' for L in LABEL_SETS] + [f'prompt:both[{L}]' for L in LABEL_SETS]

def build_features(Emb):
    B = prompt_blocks(Emb)
    F = {'image_only': Emb, 'img+colour': np.hstack([Emb, B['colour']])}
    for L in LABEL_SETS:
        F[f'img+morph[{L}]'] = np.hstack([Emb, B[f'morph[{L}]']])
        F[f'img+both[{L}]'] = np.hstack([Emb, B[f'both[{L}]']])
    for k in B:
        F[f'prompt:{k}'] = B[k]
    return F

FEATS = {r: build_features(E[r]) for r in ['colour', 'colourless']}
if RUN_GREYSCALE:
    FEATS['colour_greyscale'] = {'image_only': E['colour_greyscale']}

def make_clf():
    lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED)
    return make_pipeline(StandardScaler(), lr) if STANDARDISE else lr

def fit_predict(Xtr, ytr, Xte):
    clf = make_clf()
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1]

def mcc_at(y, p, t):
    return metrics(y, (p >= t).astype(int))['MCC']

def inner_oof(X, groups, idx):
    yi, gi = Y[idx], groups[idx]
    k = min(N_INNER, len(np.unique(gi)))
    try:
        splits = list(StratifiedGroupKFold(k, shuffle=True, random_state=SEED).split(idx, yi, gi))
    except ValueError:
        splits = list(StratifiedKFold(k, shuffle=True, random_state=SEED).split(idx, yi))
    p = np.full(len(idx), np.nan)
    for a, b in splits:
        if len(np.unique(yi[a])) < 2:
            continue
        p[b] = fit_predict(X[idx[a]], yi[a], X[idx[b]])
    return p

def choose_thresholds(p, y):
    ok = ~np.isnan(p); p, y = p[ok], y[ok]
    cands = np.unique(np.quantile(p, np.linspace(0, 1, 401)))
    t_mcc = max(cands, key=lambda t: mcc_at(y, p, t))
    pos = np.sort(p[y == 1])
    t_s90 = pos[int(np.floor(0.10 * len(pos)))] if len(pos) else 0.5
    return t_mcc, t_s90, mcc_at(y, p, 0.5)

OOF = {}
with timer('probes'):
    for scheme, folds in SCHEMES.items():
        groups = GROUPS[scheme]
        for r, F in FEATS.items():
            for cfg, X in F.items():
                inner = RUN_INNER and scheme == PRIMARY and not cfg.startswith('prompt:')
                prob = np.full(N, np.nan)
                rec = {'prob': prob}
                if inner:
                    rec.update(pred_inner_mcc=np.zeros(N, int), pred_inner_sens90=np.zeros(N, int),
                               inner_mcc05={}, thresholds={})
                for k in range(N_SPLITS):
                    te = folds == k; tr = ~te
                    prob[te] = fit_predict(X[tr], Y[tr], X[te])
                    if inner:
                        pi = inner_oof(X, groups, np.where(tr)[0])
                        t_mcc, t_s90, m05 = choose_thresholds(pi, Y[tr])
                        rec['pred_inner_mcc'][te] = prob[te] >= t_mcc
                        rec['pred_inner_sens90'][te] = prob[te] >= t_s90
                        rec['inner_mcc05'][k] = m05
                        rec['thresholds'][k] = (t_mcc, t_s90)
                OOF[(r, scheme, cfg)] = rec
            print(f'  done: {scheme} / {r}')

In [ ]:
# Nested prompt selection (R1.6): in each outer fold pick the context configuration with the best
# inner-CV MCC (training data only), then score it on the outer test fold
NESTED = []
if RUN_INNER:
    for r in ['colour', 'colourless']:
        prob = np.full(N, np.nan)
        for k in range(N_SPLITS):
            best, best_m = 'image_only', OOF[(r, PRIMARY, 'image_only')]['inner_mcc05'][k]
            for c in CONTEXT[1:]:
                m_ = OOF[(r, PRIMARY, c)]['inner_mcc05'][k]
                if m_ > best_m:
                    best, best_m = c, m_
            te = SCHEMES[PRIMARY] == k
            prob[te] = OOF[(r, PRIMARY, best)]['prob'][te]
            NESTED.append({'rendering': r, 'fold': k + 1, 'selected': best, 'inner_MCC': best_m})
        OOF[(r, PRIMARY, 'nested_selected')] = {'prob': prob}
    NESTED = pd.DataFrame(NESTED)
    display(NESTED)
    save(NESTED, 'nested_selection.csv')

## 7. Baseline: QT and heart rate only (R1.4, R2.23)

In [ ]:
if GROUPS_AVAILABLE and MATCHED[['QT', 'HR']].notna().all(1).mean() > 0.95:
    Xq = MATCHED[['QT', 'HR']].values.astype(float)
    Xq = np.where(np.isnan(Xq), np.nanmedian(Xq, 0), Xq)
    for scheme, folds in SCHEMES.items():
        prob = np.full(N, np.nan)
        for k in range(N_SPLITS):
            te = folds == k
            prob[te] = fit_predict(Xq[~te], Y[~te], Xq[te])
        OOF[('QT+HR', scheme, 'logistic')] = {'prob': prob}
    print('QT/HR baseline done. Note: applying the nomogram itself reproduces the labels by construction (MCC = 1).')
else:
    print('QT/HR baseline skipped (no QT/RR values).')

## 8. Results tables

In [ ]:
def op_points(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    spec_at_sens90 = (1 - fpr[tpr >= 0.90]).max() if (tpr >= 0.90).any() else np.nan
    sens_at_spec90 = tpr[(1 - fpr) >= 0.90].max() if ((1 - fpr) >= 0.90).any() else np.nan
    return {'Spec_at_Sens0.90': spec_at_sens90, 'Sens_at_Spec0.90': sens_at_spec90}

pooled, perfold = [], []
for (r, s, c), rec in OOF.items():
    p = rec['prob']
    base = {'rendering': r, 'scheme': s, 'config': c}
    pooled.append({**base, 'threshold': '0.5 (balanced weights)', **metrics(Y, p >= 0.5, p), **op_points(Y, p)})
    if 'pred_inner_mcc' in rec:
        pooled.append({**base, 'threshold': 'inner CV: max MCC', **metrics(Y, rec['pred_inner_mcc'], p)})
        pooled.append({**base, 'threshold': 'inner CV: sens >= 0.90', **metrics(Y, rec['pred_inner_sens90'], p)})
    for k in range(N_SPLITS):
        te = SCHEMES[s] == k
        perfold.append({**base, 'fold': k + 1, **metrics(Y[te], p[te] >= 0.5, p[te])})
POOLED = pd.DataFrame(pooled)
PERFOLD = pd.DataFrame(perfold)
FOLDMEAN = (PERFOLD.groupby(['rendering', 'scheme', 'config'])[['Sens', 'Spec', 'BA', 'PPV', 'MCC', 'ROC_AUC', 'AUPRC']]
            .agg(['mean', 'std']).round(4))
FOLDMEAN.columns = [f'{a}_{b}' for a, b in FOLDMEAN.columns]
FOLDMEAN = FOLDMEAN.reset_index()
save(POOLED, 'probe_pooled_out_of_fold.csv')
save(PERFOLD, 'probe_per_fold.csv')
save(FOLDMEAN, 'probe_fold_mean_sd.csv')

oof_long = pd.concat([pd.DataFrame({'file_name': MATCHED.file_name, 'y': Y, 'rendering': r, 'scheme': s,
                                    'config': c, 'fold': SCHEMES[s] + 1, 'prob': rec['prob']})
                      for (r, s, c), rec in OOF.items()], ignore_index=True)
oof_long.to_csv(os.path.join(OUT_DIR, 'oof_predictions.csv.gz'), index=False)

show = ['rendering', 'scheme', 'config', 'Sens', 'Spec', 'BA', 'PPV', 'MCC', 'ROC_AUC', 'AUPRC', 'Spec_at_Sens0.90']
print('Main configurations, pooled out-of-fold, threshold 0.5:')
key = POOLED[(POOLED.threshold.str.startswith('0.5')) &
             (POOLED.config.isin(['image_only', 'nested_selected', 'logistic', 'img+morph[L3]', 'img+both[L3]']))]
display(key[show].sort_values(['scheme', 'rendering', 'config']).round(3))

## 9. Participant-cluster bootstrap for paired differences (R1.6)

In [ ]:
clusters = MATCHED.participant.values if GROUPS_AVAILABLE else np.arange(N)
uniq = np.unique(clusters)
idx_by = {g: np.where(clusters == g)[0] for g in uniq}
rng_b = np.random.default_rng(SEED)
BOOT = [np.concatenate([idx_by[g] for g in rng_b.choice(uniq, len(uniq), replace=True)]) for _ in range(N_BOOT)]

MET = {
    'MCC': lambda y, p: metrics(y, p >= 0.5)['MCC'],
    'BA': lambda y, p: metrics(y, p >= 0.5)['BA'],
    'ROC_AUC': fast_auc,
    'AUPRC': lambda y, p: average_precision_score(y, p) if 0 < y.sum() < len(y) else np.nan,
}
_cache = {}
def dist(key, m):
    if (key, m) not in _cache:
        p = OOF[key]['prob']
        _cache[(key, m)] = (MET[m](Y, p), np.array([MET[m](Y[i], p[i]) for i in BOOT]))
    return _cache[(key, m)]

def compare(family, label, keys_a, keys_b, m):
    pa = np.mean([dist(k, m)[0] for k in keys_a]); pb = np.mean([dist(k, m)[0] for k in keys_b])
    da = np.mean([dist(k, m)[1] for k in keys_a], 0); db = np.mean([dist(k, m)[1] for k in keys_b], 0)
    d = da - db
    d = d[~np.isnan(d)]
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return {'family': family, 'comparison': label, 'metric': m, 'A': pa, 'B': pb, 'diff_A_minus_B': pa - pb,
            'CI95_low': np.percentile(d, 2.5), 'CI95_high': np.percentile(d, 97.5), 'p_boot': p}

with timer('bootstrap'):
    rows = []
    S = PRIMARY
    for m in MET:
        rows.append(compare('rendering', 'colour vs colourless (image only)',
                            [('colour', S, 'image_only')], [('colourless', S, 'image_only')], m))
        if RUN_GREYSCALE:
            rows.append(compare('rendering', 'colour vs greyscale-of-colour (image only)',
                                [('colour', S, 'image_only')], [('colour_greyscale', S, 'image_only')], m))
            rows.append(compare('rendering', 'greyscale-of-colour vs colourless (image only)',
                                [('colour_greyscale', S, 'image_only')], [('colourless', S, 'image_only')], m))
        if ('QT+HR', S, 'logistic') in OOF:
            rows.append(compare('baseline', 'colour image probe vs QT+HR logistic',
                                [('colour', S, 'image_only')], [('QT+HR', S, 'logistic')], m))
        for r in ['colour', 'colourless']:
            for c in CONTEXT[1:] + (['nested_selected'] if RUN_INNER else []):
                rows.append(compare(f'context vs image only ({r})', f'{c} vs image_only',
                                    [(r, S, c)], [(r, S, 'image_only')], m))
            # 2x2 as in the paper's Table 3: colour named in prompt vs not named, pooled over label sets
            named = [(r, S, 'img+colour')] * 4 + [(r, S, f'img+both[{L}]') for L in LABEL_SETS]
            not_named = [(r, S, 'image_only')] * 4 + [(r, S, f'img+morph[{L}]') for L in LABEL_SETS]
            rows.append(compare('2x2 prompt effect', f'colour named vs not named ({r})', named, not_named, m))
        rows.append(compare('2x2 image effect', 'colour vs colourless, pooled over all context configs',
                            [('colour', S, c) for c in CONTEXT], [('colourless', S, c) for c in CONTEXT], m))
    BOOTRES = pd.DataFrame(rows)

def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p, dtype=float); run = 0
    for i, j in enumerate(o):
        run = max(run, (len(p) - i) * p[j]); adj[j] = min(1.0, run)
    return adj
BOOTRES['p_holm'] = np.nan
for (fam, m), g in BOOTRES.groupby(['family', 'metric']):
    BOOTRES.loc[g.index, 'p_holm'] = holm(g.p_boot.values)
save(BOOTRES, 'bootstrap_paired_differences.csv')
display(BOOTRES[BOOTRES.metric == 'MCC'].round(4))

## 10. Third condition: the submitted paper's stretched images (R1.4)

In [ ]:
# Old images were stretched from R to R (no absolute time). If their numbers are the same recording numbers,
# compare on the same recordings and folds:
#   stretched colourless (no absolute time) | fixed-axis colourless (time, no threshold) | fixed-axis colour (time + threshold)
from sklearn.metrics import cohen_kappa_score
THIRD = None
if OLD_INV:
    rec_old = {r: OLD_INV[r].assign(rec=OLD_INV[r].file_name.str.extract(r'^(\d+)')[0].astype(float))
               for r in OLD_INV}
    base = MATCHED[['rec', 'y']].reset_index().rename(columns={'index': 'i_new'})
    ok_link, links = True, {}
    for r, df in rec_old.items():
        df = df.dropna(subset=['rec']).assign(rec=lambda d: d.rec.astype(int)).reset_index().rename(columns={'index': 'i_old'})
        lk = base.merge(df[['rec', 'i_old', 'y']].rename(columns={'y': 'y_old'}), on='rec')
        kappa = cohen_kappa_score(lk.y, lk.y_old) if len(lk) and lk.y.nunique() > 1 else np.nan
        print(f'old {r}: {len(lk)} of {N} recordings linked; label agreement kappa = {kappa:.3f}')
        if not (len(lk) >= 0.5 * N and kappa >= 0.8):
            ok_link = False
        links[r] = lk
    if ok_link:
        common = set(links['colour'].i_new) & set(links['colourless'].i_new)
        idx = np.array(sorted(common))
        old_pos = {r: links[r].set_index('i_new').loc[idx, 'i_old'].values for r in links}
        conds = {'colourless, stretched R-R (old)': E_OLD['colourless'][old_pos['colourless']],
                 'colour, stretched R-R (old)': E_OLD['colour'][old_pos['colour']],
                 'colourless, fixed axis (new)': E['colourless'][idx],
                 'colour, fixed axis (new)': E['colour'][idx]}
        y3, f3, g3 = Y[idx], SCHEMES[PRIMARY][idx], MATCHED.participant.values[idx]
        P3 = {}
        for name, X in conds.items():
            p = np.full(len(idx), np.nan)
            for k in range(N_SPLITS):
                te = f3 == k
                p[te] = fit_predict(X[~te], y3[~te], X[te])
            P3[name] = p
        rows = [{'condition': n, 'n': len(idx), 'high_risk': int(y3.sum()), **metrics(y3, p >= 0.5, p)} for n, p in P3.items()]
        THIRD = pd.DataFrame(rows)
        display(THIRD.round(3))
        save(THIRD, 'third_condition_metrics.csv')

        uq = np.unique(g3); ib = {g: np.where(g3 == g)[0] for g in uq}
        rb = np.random.default_rng(SEED)
        B3 = [np.concatenate([ib[g] for g in rb.choice(uq, len(uq), replace=True)]) for _ in range(N_BOOT)]
        comp = [('colourless, fixed axis (new)', 'colourless, stretched R-R (old)'),
                ('colour, fixed axis (new)', 'colour, stretched R-R (old)'),
                ('colour, fixed axis (new)', 'colourless, fixed axis (new)'),
                ('colour, stretched R-R (old)', 'colourless, stretched R-R (old)')]
        out = []
        for a, b in comp:
            for mname, fn in MET.items():
                d = np.array([fn(y3[i], P3[a][i]) - fn(y3[i], P3[b][i]) for i in B3])
                d = d[~np.isnan(d)]
                out.append({'A': a, 'B': b, 'metric': mname, 'A_value': fn(y3, P3[a]), 'B_value': fn(y3, P3[b]),
                            'diff': fn(y3, P3[a]) - fn(y3, P3[b]),
                            'CI95_low': np.percentile(d, 2.5), 'CI95_high': np.percentile(d, 97.5),
                            'p_boot': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))})
        THIRD_BOOT = pd.DataFrame(out)
        display(THIRD_BOOT[THIRD_BOOT.metric == 'MCC'].round(4))
        save(THIRD_BOOT, 'third_condition_bootstrap.csv')
    else:
        print('Old image numbers do not link reliably to the new recording numbers; third condition skipped.')
else:
    print('Old images not available; third condition skipped.')

## 11. Check: the original protocol on the old images reproduces the published numbers (R1.8)

In [ ]:
EXPECTED_ZS = {('colour', 'colour only (C+M̄)', 'No Label'): (175, 6, 972, 3882),
               ('colour', 'morphology only (C̄+M)', 'L3'): (24, 157, 4408, 446),
               ('colourless', 'colour only (C+M̄)', 'No Label'): (181, 0, 0, 4573),
               ('colourless', 'morphology only (C̄+M)', 'L3'): (84, 97, 2568, 2005)}
EXPECTED_TRAINED = {('colour', 'none[No Label]'): 0.437, ('colour', 'morph[L3]'): 0.446,
                    ('colourless', 'none[No Label]'): 0.312, ('colourless', 'both[L3]'): 0.328}
rep = []
for r in OLD_INV:
    yf = OLD_INV[r].y.values
    Ef = E_OLD[r]
    okf = ~np.isnan(Ef).any(1)
    rep.append({'check': f'old {r}: images (high-risk / low-risk)', 'published': '181 / 4854' if r == 'colour' else '181 / 4573',
                'rerun': f'{int(yf[okf].sum())} / {int((yf[okf] == 0).sum())}'})
    zs = zero_shot_table(Ef[okf], yf[okf])
    for (rr_, strat, L), exp in EXPECTED_ZS.items():
        if rr_ != r:
            continue
        row = zs[(zs.strategy == strat) & (zs.label_set == L)].iloc[0]
        rep.append({'check': f'zero-shot {r} {L} {strat} (TP,FN,TN,FP)', 'published': str(exp),
                    'rerun': str(tuple(int(v) for v in (row.TP, row.FN, row.TN, row.FP)))})
    B = prompt_blocks(Ef[okf])
    blocks = {'none[No Label]': np.full((okf.sum(), 2), 0.5), 'morph[L3]': B['morph[L3]'], 'both[L3]': B['both[L3]']}
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for (rr_, bname), exp in EXPECTED_TRAINED.items():
        if rr_ != r:
            continue
        X = np.hstack([Ef[okf], blocks[bname]])
        mccs = []
        for tr, te in skf.split(X, yf[okf]):
            clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            clf.fit(X[tr], yf[okf][tr])
            mccs.append(metrics(yf[okf][te], clf.predict(X[te]))['MCC'])
        rep.append({'check': f'trained {r} {bname}, fold-mean MCC (original protocol)', 'published': exp,
                    'rerun': round(float(np.mean(mccs)), 3)})
REPRO = pd.DataFrame(rep)
display(REPRO)
if len(REPRO):
    save(REPRO, 'reproduction_check_old_images.csv')

## 12. Figures and logs

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
curves = [(('colour', PRIMARY, 'image_only'), 'Coloured, image only'),
          (('colourless', PRIMARY, 'image_only'), 'Colourless, image only'),
          (('colour_greyscale', PRIMARY, 'image_only'), 'Coloured shown in greyscale'),
          (('QT+HR', PRIMARY, 'logistic'), 'QT + HR logistic')]
for key, lab in curves:
    if key not in OOF:
        continue
    p = OOF[key]['prob']
    pr, rc, _ = precision_recall_curve(Y, p)
    axes[0].plot(rc, pr, label=f'{lab} (AUPRC {average_precision_score(Y, p):.2f})')
    fpr, tpr, _ = roc_curve(Y, p)
    axes[1].plot(fpr, tpr, label=f'{lab} (AUC {fast_auc(Y, p):.3f})')
axes[0].axhline(Y.mean(), color='grey', ls=':', lw=1)
axes[0].set(xlabel='Recall (sensitivity)', ylabel='Precision (PPV)', title=f'Precision-recall, {PRIMARY} CV')
axes[1].plot([0, 1], [0, 1], color='grey', ls=':', lw=1)
axes[1].set(xlabel='1 - specificity', ylabel='Sensitivity', title=f'ROC, {PRIMARY} CV')
for a in axes:
    a.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'pr_roc_curves.png'), dpi=200)
plt.show()

ENV['n_matched'] = int(N); ENV['n_high_risk'] = int(Y.sum()); ENV['primary_scheme'] = PRIMARY
ENV['participants'] = int(MATCHED.participant.nunique()) if GROUPS_AVAILABLE else None
with open(os.path.join(OUT_DIR, 'environment.json'), 'w') as f:
    json.dump(ENV, f, indent=2, default=str)
save(pd.DataFrame(TIMINGS), 'timings.csv')
print('All outputs in', OUT_DIR)